In [ ]:
%pip install gym
%pip install tensorflow
%pip install InquirerPy




TODO:
- Adjust learning rate
- Add metrics like entroy or accuracy but make sure to apply to actor and critic
- Come up with a better system of analyzing results and comparing them


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Dense, concatenate

from collections import deque 


import gym
from gym import spaces
import gym_game
import numpy as np
import math

class A2CAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.value_size = 1

        self.actor = self.build_actor()
        self.critic = self.build_critic()

    def build_actor(self):
        state_input = Input(shape=(self.state_size,))
        advantages = Input(shape=(1,))

        x = Dense(128, activation='relu')(state_input)
        x = Dense(64, activation='relu')(x)
        action_prob = Dense(self.action_size, activation='softmax')(x)
        # action_prob = Dense(self.action_size, activation='softmax', temperature=1.0)(x)


        actor = Model(inputs=[state_input, advantages], outputs=[action_prob])
        actor.compile(optimizer=Adam(lr=0.001), loss='categorical_crossentropy')
        return actor
    

    def build_critic(self):
        state_input = Input(shape=(self.state_size,))

        x = Dense(128, activation='relu')(state_input)
        x = Dense(64, activation='relu')(x)
        value = Dense(self.value_size, activation='linear')(x)

        critic = Model(inputs=[state_input], outputs=[value])
        critic.compile(optimizer=Adam(lr=0.001), loss='mse')
        return critic

    def act(self, state):
        probabilities = self.actor.predict([state, np.zeros((1, 1))])
        action = np.random.choice(self.action_size, p=np.squeeze(probabilities))
        return action

    def get_value(self, state):
        return self.critic.predict(state)[0]

    def train(self, state, action, reward, next_state, done):
        target = reward + 0.95 * self.get_value(next_state) * (1 - done)
        advantages = target - self.get_value(state)

        action_one_hot = np.zeros([self.action_size])
        action_one_hot[action] = 1

        self.actor.fit([state, advantages], [action_one_hot.reshape([1, self.action_size])], epochs=1, verbose=0)
        self.critic.fit(state, target.reshape([1, 1]), epochs=1, verbose=0)

# Assuming JumbleBerryEnv is your custom environment
env = gym.make('gym_game/Jumblegame-v1')

state_space = len(tuple(space.n if isinstance(space, gym.spaces.Discrete) else space.shape[0] for space in env.observation_space.spaces))
action_space = env.action_space.n

agent = A2CAgent(state_space, action_space)

# Training loop
episodes = 50
for episode in range(episodes):
    state, _ = env.reset()
    state = np.reshape(state, [1, state_space])
    print(state)

    total_reward = 0
    done = False

    action_list = []

    while not done:
        # print("State:")
        # print(state)
        # print(f"Episode: {episode}, State: {state}")
        action = agent.act(state)
        action_list.append(action)
        next_state, reward, done, _ = env.step(action)
        next_state = np.reshape(next_state, [1, state_space])

        agent.train(state, action, reward, next_state, done)

        state = next_state
        total_reward += reward

    print(f"Episode: {episode + 1}, Total Reward: {total_reward}")
    print(state)
    print(action_list)


The following will stop at 2-4 actions and always repeat:

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Dense, concatenate

from collections import deque 


import gym
from gym import spaces
import gym_game
import numpy as np
import math

class A2CAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.value_size = 1

        self.actor = self.build_actor()
        self.critic = self.build_critic()

    def build_actor(self):
        state_input = Input(shape=(self.state_size,))
        advantages = Input(shape=(1,))

        x = Dense(128, activation='relu')(state_input)
        x = Dense(64, activation='relu')(x)
        action_prob = Dense(self.action_size, activation='softmax')(x)

        actor = Model(inputs=[state_input, advantages], outputs=[action_prob])
        actor.compile(optimizer=Adam(lr=0.001), loss='categorical_crossentropy', metrics=['entropy'])
        return actor

    def build_critic(self):
        state_input = Input(shape=(self.state_size,))

        x = Dense(128, activation='relu')(state_input)
        x = Dense(64, activation='relu')(x)
        value = Dense(self.value_size, activation='linear')(x)

        critic = Model(inputs=[state_input], outputs=[value])
        critic.compile(optimizer=Adam(lr=0.001), loss='mse')
        return critic

    def act(self, state):
        probabilities = self.actor.predict([state, np.zeros((1, 1))])
        action = np.random.choice(self.action_size, p=np.squeeze(probabilities))
        return action

    def get_value(self, state):
        return self.critic.predict(state)[0]

    def train(self, state, action, reward, next_state, done):
        target = reward + 0.95 * self.get_value(next_state) * (1 - done)
        advantages = target - self.get_value(state)

        action_one_hot = np.zeros([self.action_size])
        action_one_hot[action] = 1

        self.actor.fit([state, advantages], [action_one_hot.reshape([1, self.action_size])], epochs=1, verbose=0)
        self.critic.fit(state, target.reshape([1, 1]), epochs=1, verbose=0)

# Assuming JumbleBerryEnv is your custom environment
env = gym.make('gym_game/Jumblegame-v1')

state_space = len(tuple(space.n if isinstance(space, gym.spaces.Discrete) else space.shape[0] for space in env.observation_space.spaces))
action_space = env.action_space.n

agent = A2CAgent(state_space, action_space)

# Training loop
episodes = 1000
for episode in range(episodes):
    state, _ = env.reset()
    state = np.reshape(state, [1, state_space])
    print(state)

    total_reward = 0
    done = False

    action_list = []

    while not done:
        # print("State:")
        # print(state)
        # print(f"Episode: {episode}, State: {state}")
        action = agent.act(state)
        action_list.append(action)
        next_state, reward, done, _ = env.step(action)
        next_state = np.reshape(next_state, [1, state_space])

        agent.train(state, action, reward, next_state, done)

        state = next_state
        total_reward += reward

    print(f"Episode: {episode + 1}, Total Reward: {total_reward}")
    print(state)
    print(action_list)


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Dense, concatenate

from collections import deque 


import gym
from gym import spaces
import gym_game
import numpy as np
import math

In [ ]:
# Assuming JumbleBerryEnv is your custom environment
env = gym.make('gym_game/Jumblegame-v1')

input_shape = len(env.observation_space)
num_actions = env.action_space.n

In [ ]:
# Define input layer
inputs = Input(shape=(input_shape,))

# Define shared hidden layers
hidden1 = Dense(32, activation='relu')(inputs)
hidden2 = Dense(32, activation='relu')(hidden1)

# Define separate output layers
output1 = Dense(num_actions)(hidden2)
output2 = Dense(num_actions, activation='softmax')(hidden2)

model = tf.keras.Model(inputs=inputs, outputs=[output2, output1])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

In [ ]:
num_episodes = 10000
gamma = tf.cast(tf.constant(0.9),tf.float64)
count = 0
n = 5 #keep n very small
min_loss = math.inf

maxlen = n
states = deque(maxlen=maxlen)
actions = deque(maxlen=maxlen)
rewards = deque(maxlen=maxlen)
next_states = deque(maxlen=maxlen)
dones = deque(maxlen=maxlen)

In [ ]:
for episode in range(num_episodes):
    # Reset the environment and get the initial state
    state = list(env.reset())
    state[2] = 1 if state[2] else 0

    # Keep track of the states, actions, and rewards for each step in the episode

    # Run the episode
    while True:
        count=count+1
        # Get the action probabilities from the policy network
        action_probs, _ = model.predict(np.array([state]),verbose=0)

        # Choose an action based on the action probabilities
        action = np.random.choice(num_actions, p=action_probs[0])

        # Take the chosen action and observe the next state and reward
        next_state, reward, done, _ = env.step(action)
        next_state = list(next_state)
        next_state[2] = 1 if next_state[2] else 0
        # Store the current state, action, and reward
        states.append(state)
        actions.append(action)
        rewards.append(reward)
        next_states.append(next_state)
        dones.append(1 if done else 0)

        state = next_state

        # End the episode if the environment is done
        if done:
            done=False
            break

    if count>n:
            count=0
            # Convert the lists of states, actions, and discounted rewards to tensors
            states_ = tf.convert_to_tensor(states)
            actions_ = tf.convert_to_tensor(actions)
            rewards_ = tf.convert_to_tensor(rewards)
            next_states_ = tf.convert_to_tensor(next_states)

     

            with tf.GradientTape(persistent=True) as tape:
       
                action_probs,q_values1 = model(states_)
                q_value1 = tf.cast(tf.gather(q_values1,actions_,axis=1,batch_dims=1),tf.float64)
                _,q_value2 = model(next_states_)
                q_value2 = tf.cast(tf.reduce_max(q_value2),tf.float64)
                
                target_value = tf.cast(rewards_,tf.float64) + (tf.cast(1,tf.float64)-tf.cast(dones,tf.float64))*gamma*q_value2
                value_loss = tf.reduce_mean(tf.math.pow(target_value-q_value1,2))
                advantage = target_value-q_value1
      
                action_probs = tf.cast(tf.math.log(tf.gather(action_probs,actions,axis=1,batch_dims=1)),tf.float64)
                loss = -tf.reduce_mean(action_probs * advantage) + value_loss


            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))

    
            print('Episode {} done with loss {} !!!!!!'.format(episode, loss))
            if loss<min_loss:
                min_loss = loss
                model.save('blackjack/')

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Dense, concatenate

from collections import deque 


import gym
from gym import spaces
import gym_game
import numpy as np
import math

class A2CAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.value_size = 1

        self.actor = self.build_actor()
        self.critic = self.build_critic()

    def build_actor(self):
        state_input = Input(shape=(self.state_size,))
        advantages = Input(shape=(1,))

        x = Dense(128, activation='relu')(state_input)
        x = Dense(64, activation='relu')(x)
        action_prob = Dense(self.action_size, activation='softmax')(x)

        actor = Model(inputs=[state_input, advantages], outputs=[action_prob])
        actor.compile(optimizer=Adam(lr=0.001), loss='categorical_crossentropy')
        return actor

    def build_critic(self):
        state_input = Input(shape=(self.state_size,))

        x = Dense(128, activation='relu')(state_input)
        x = Dense(64, activation='relu')(x)
        value = Dense(self.value_size, activation='linear')(x)

        critic = Model(inputs=[state_input], outputs=[value])
        critic.compile(optimizer=Adam(lr=0.001), loss='mse')
        return critic

    def act(self, state):
        probabilities = self.actor.predict([state, np.zeros((1, 1))])
        action = np.random.choice(self.action_size, p=np.squeeze(probabilities))
        return action

    def get_value(self, state):
        return self.critic.predict(state)[0]

    def train(self, state, action, reward, next_state, done):
        target = reward + 0.95 * self.get_value(next_state) * (1 - done)
        advantages = target - self.get_value(state)

        action_one_hot = np.zeros([self.action_size])
        action_one_hot[action] = 1

        self.actor.fit([state, advantages], [action_one_hot.reshape([1, self.action_size])], epochs=1, verbose=0)
        self.critic.fit(state, target.reshape([1, 1]), epochs=1, verbose=0)

# Assuming JumbleBerryEnv is your custom environment
env = gym.make('gym_game/Jumblegame-v1')

state_space = len(tuple(space.n if isinstance(space, gym.spaces.Discrete) else space.shape[0] for space in env.observation_space.spaces))
action_space = env.action_space.n

agent = A2CAgent(state_space, action_space)

# Training loop
episodes = 1000
for episode in range(episodes):
    state, _ = env.reset()
    print(state)
    print(state_space)
    state = np.reshape(state, [1, state_space])


 
    total_reward = 0
    done = False

    while not done:
        action = agent.act(state)
        next_state, reward, done, _ = env.step(action)
        next_state = np.reshape(next_state, [1, state_space])

        agent.train(state, action, reward, next_state, done)

        state = next_state
        total_reward += reward

    print(f"Episode: {episode + 1}, Total Reward: {total_reward}")
